# Efficiently Computing Equilibrium Grids

Generating many pycalphad equilibrium results is a common requirement for
high-throughput workflows such as training a surrogate model, building an active
learning dataset, or screening a large composition-temperature space.

This example walks through three approaches in order of correctness and
flexibility: a naive approach that inadvertently creates a much larger grid than
intended, the proper use of `equilibrium` for regular broadcast grids, and the
`Workspace` approach for explicit simplex point designs. All three approaches are
timed so you can see the practical difference.

In [ ]:
from itertools import product
from time import perf_counter
from importlib.resources import files

import numpy as np
import pandas as pd

from pycalphad import Database, Workspace, equilibrium, variables as v
import pycalphad.tests.databases

dbf = Database(files(pycalphad.tests.databases).joinpath("COST507.tdb"))
components = ["AL", "MG", "SI", "CU", "VA"]
elements = components[:-1]
solutes = elements[1:]
phases = list(dbf.phases.keys())

temperatures = [1000.0, 1250.0, 1500.0]
pressure = 101325.0
density = 4
eps = 1e-4

## Approach 1 (Naive): Passing Simplex Columns as Independent Broadcast Axes

A natural but incorrect first attempt is to generate a simplex design, extract
each solute column, and pass the columns directly to `equilibrium` as separate
condition arrays.

`equilibrium` treats each array as an independent axis and broadcasts them
against one another, so if each solute array has 10 entries, pycalphad evaluates
a `10 × 10 × 10` composition grid before temperature is included. Many of those
broadcast compositions are infeasible because the solute fractions sum to more
than one — the intended point design has been silently inflated into a much
larger and partially invalid grid.

In [ ]:
naive_conditions = {
    v.T: temperatures,
    v.P: pressure,
    v.N: 1.0,
    v.X("MG"): np.array([
        9.99700090e-05, 9.99800040e-05, 9.99900010e-05,
        2.49950010e-01, 2.49975002e-01, 2.50000000e-01,
        4.99900020e-01, 4.99950005e-01,
        7.49850030e-01, 9.99700090e-01,
    ]),
    v.X("SI"): np.array([
        9.99700090e-05, 9.99800040e-05, 9.99900010e-05,
        2.49950010e-01, 2.49975002e-01, 2.50000000e-01,
        4.99900020e-01, 4.99950005e-01,
        7.49850030e-01, 9.99700090e-01,
    ]),
    v.X("CU"): np.array([
        9.99700090e-05, 9.99800040e-05, 9.99900010e-05,
        2.49950010e-01, 2.49975002e-01, 2.50000000e-01,
        4.99900020e-01, 4.99950005e-01,
        7.49850030e-01, 9.99700090e-01,
    ]),
}

axis_lengths = {
    str(key): len(value)
    for key, value in naive_conditions.items()
    if np.ndim(value) > 0
}
broadcast_point_count = int(np.prod(list(axis_lengths.values())))
print(f"Axis lengths: {axis_lengths}")
print(f"Broadcast grid size: {broadcast_point_count} points (many infeasible)")

start = perf_counter()
naive_result = equilibrium(dbf, components, phases, naive_conditions)
naive_seconds = perf_counter() - start
print(f"\nNaive equilibrium: {naive_seconds:.2f} s")
naive_result

## Approach 2: Using `equilibrium` Correctly for a Regular Broadcast Grid

If the desired design really is a regular grid, pass the independent axis levels
directly using `np.linspace`. `equilibrium` then constructs the broadcast grid
from those clean axis values.

The grid still includes infeasible compositions where
`X(MG) + X(SI) + X(CU) > 1`, so those entries will be `NaN` and should be
masked or dropped afterward. For a true simplex design, see Approach 3.

In [ ]:
regular_grid_conditions = {
    v.T: temperatures,
    v.P: pressure,
    v.N: 1.0,
    v.X("MG"): np.linspace(eps, 1 - len(solutes) * eps, density + 1),
    v.X("SI"): np.linspace(eps, 1 - len(solutes) * eps, density + 1),
    v.X("CU"): np.linspace(eps, 1 - len(solutes) * eps, density + 1),
}

start = perf_counter()
regular_grid_result = equilibrium(dbf, components, phases, regular_grid_conditions)
regular_grid_seconds = perf_counter() - start
print(f"equilibrium (regular broadcast grid): {regular_grid_seconds:.2f} s")
regular_grid_result

## Approach 3 (Recommended): Using `Workspace` for Explicit Simplex Points

For surrogate training, active learning, or any workflow driven by a named
design, the evaluation target is a list of valid compositions rather than a
rectangular grid. In that case, build explicit simplex rows and evaluate exactly
those points with a reused `Workspace`.

A reused `Workspace` preserves expensive internal pycalphad objects (models,
phase-record factory) across points. When the condition keys are unchanged
between calls, only the equilibrium calculation itself reruns — not the full
setup — which gives a significant speedup over repeated `equilibrium` calls.

The output is a flat table of one row per (composition, temperature) point with
no infeasible entries, making it directly usable as training data.

In [ ]:
def composition_grid(component_count, density):
    rows = []
    for counts in product(range(density + 1), repeat=component_count):
        if sum(counts) == density:
            row = np.asarray(counts, dtype=float) / density
            row = np.clip(row, eps, 1.0 - eps * len(solutes))
            rows.append(row / row.sum())
    return np.asarray(rows)


def drop_empty_columns(frame):
    return frame.dropna(axis="columns", how="all")


composition_rows = composition_grid(len(elements), density)
composition_conditions = [
    {v.X(solute): amount for solute, amount in zip(solutes, composition_row[1:])}
    for composition_row in composition_rows
]
n_points = len(composition_rows) * len(temperatures)
print(f"Evaluating {len(composition_rows)} simplex points × {len(temperatures)} temperatures = {n_points} total")

data_rows = []
start = perf_counter()
wks = Workspace(dbf, components, phases)

for temperature in temperatures:
    for comp_conds in composition_conditions:
        conditions = {v.T: temperature, v.P: pressure, v.N: 1, **comp_conds}
        wks.conditions = conditions
        out = {
            str(key): value[()]
            for key, value in wks.get_dict("T", "X(*)", "NP(*)", "X(*,*)").items()
        }
        data_rows.append(out)

workspace_seconds = perf_counter() - start
workspace_df = drop_empty_columns(pd.DataFrame(data_rows))
print(f"Workspace (explicit simplex points): {workspace_seconds:.2f} s")
workspace_df.head()

In [ ]:
print("Timing comparison")
print("-" * 46)
print(f"  Naive equilibrium (inflated broadcast):  {naive_seconds:>6.2f} s")
print(f"  equilibrium (correct broadcast grid):    {regular_grid_seconds:>6.2f} s")
print(f"  Workspace (explicit simplex points):     {workspace_seconds:>6.2f} s")

## Summary and Guidance

The timing comparison above shows three progressively better approaches for the
same task.

The naive approach is slowest because broadcasting simplex columns creates an
unintentionally large grid — typically many times the intended size — with a
significant fraction of infeasible compositions evaluated and discarded.

The correct `equilibrium` broadcast approach is faster because the condition
arrays contain clean, independent axis values. It is the right choice when you
genuinely want the full rectangular xarray result indexed by condition axes, and
are happy to mask or drop infeasible simplex corners afterward.

The `Workspace` approach evaluates only the valid simplex points and produces a
flat table ready for use as surrogate training data or as input to an active
learning pipeline. It is the recommended default for workflows that start from a
named design: generate a set of valid compositions, evaluate exactly those rows,
and record successes and failures explicitly.